In [10]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('./data/emotions_clean.csv') #import dataset and org in table - cleaned, lemmetized, stop words and negations

In [12]:
df.head() #show me first 5 rows 

,Unnamed: 0,text,emotion,doc,tokens,lemmas,clean_tokens,adjectives,clean_tokens_neg,word_count
0,0,i didnt feel humiliated,sadness,i didnt feel humiliated,"['i', 'did', 'nt', 'feel', 'humiliated']","['I', 'do', 'not', 'feel', 'humiliated']","['feel', 'humiliated']",['humiliated'],"['not_feel', 'humiliated']",5
1,1,i can go from feeling so hopeless to so damned...,sadness,i can go from feeling so hopeless to so damned...,"['i', 'can', 'go', 'from', 'feeling', 'so', 'h...","['I', 'can', 'go', 'from', 'feel', 'so', 'hope...","['feel', 'hopeless', 'damned', 'hopeful', 'car...","['hopeless', 'hopeful', 'awake']","['feel', 'hopeless', 'damned', 'hopeful', 'car...",21
2,2,im grabbing a minute to post i feel greedy wrong,anger,im grabbing a minute to post i feel greedy wrong,"['i', 'm', 'grabbing', 'a', 'minute', 'to', 'p...","['I', 'm', 'grab', 'a', 'minute', 'to', 'post'...","['m', 'grab', 'minute', 'post', 'feel', 'greed...","['greedy', 'wrong']","['m', 'grab', 'minute', 'post', 'feel', 'greed...",11
3,3,i am ever feeling nostalgic about the fireplac...,love,i am ever feeling nostalgic about the fireplac...,"['i', 'am', 'ever', 'feeling', 'nostalgic', 'a...","['I', 'be', 'ever', 'feel', 'nostalgic', 'abou...","['feel', 'nostalgic', 'fireplace', 'know', 'pr...",['nostalgic'],"['feel', 'nostalgic', 'fireplace', 'know', 'pr...",18
4,4,i am feeling grouchy,anger,i am feeling grouchy,"['i', 'am', 'feeling', 'grouchy']","['I', 'be', 'feel', 'grouchy']","['feel', 'grouchy']",['grouchy'],"['feel', 'grouchy']",4


In [13]:
df["emotion"].value_counts() #selects emotions column and counts unique value

emotion
joy         5362
sadness     4666
anger       2159
fear        1937
love        1304
surprise     572
Name: count, dtype: int64

In [16]:
df.shape

(16000, 10)

In [ ]:
df.tail(10) #shows last 10 rows 

,Unnamed: 0,text,emotion,doc,tokens,lemmas,clean_tokens,adjectives,clean_tokens_neg,word_count
15990,15990,i feel really glad that i dont look like the c...,joy,i feel really glad that i dont look like the c...,"['i', 'feel', 'really', 'glad', 'that', 'i', '...","['I', 'feel', 'really', 'glad', 'that', 'I', '...","['feel', 'glad', 'look', 'like', 'celebrity', ...","['glad', 'beautiful']","['feel', 'glad', 'not_look', 'like', 'celebrit...",21
15991,15991,i have wanted to perhaps convey my feelings of...,sadness,i have wanted to perhaps convey my feelings of...,"['i', 'have', 'wanted', 'to', 'perhaps', 'conv...","['I', 'have', 'want', 'to', 'perhaps', 'convey...","['want', 'convey', 'feeling', 'matter', 'inste...",['irrelevant'],"['want', 'convey', 'feeling', 'matter', 'inste...",28
15992,15992,i moved away he said something that made me fe...,anger,i moved away he said something that made me fe...,"['i', 'moved', 'away', 'he', 'said', 'somethin...","['I', 'move', 'away', 'he', 'say', 'something'...","['away', 'feel', 'violent']",['violent'],"['move', 'away', 'say', 'feel', 'violent', 'no...",20
15993,15993,i most days feel like if braeden and calvin ar...,joy,i most days feel like if braeden and calvin ar...,"['i', 'most', 'days', 'feel', 'like', 'if', 'b...","['I', 'most', 'day', 'feel', 'like', 'if', 'br...","['day', 'feel', 'like', 'braeden', 'calvin', '...","['most', 'happy', 'successful']","['day', 'feel', 'like', 'braeden', 'calvin', '...",18
15994,15994,i feel so sympathetic empathetic towards them,love,i feel so sympathetic empathetic towards them,"['i', 'feel', 'so', 'sympathetic', 'empathetic...","['I', 'feel', 'so', 'sympathetic', 'empathetic...","['feel', 'sympathetic', 'empathetic']",['sympathetic'],"['feel', 'sympathetic', 'empathetic']",7
15995,15995,i just had a very brief time in the beanbag an...,sadness,i just had a very brief time in the beanbag an...,"['i', 'just', 'had', 'a', 'very', 'brief', 'ti...","['I', 'just', 'have', 'a', 'very', 'brief', 't...","['brief', 'time', 'beanbag', 'anna', 'feel', '...",['brief'],"['brief', 'time', 'beanbag', 'say', 'anna', 'f...",24
15996,15996,i am now turning and i feel pathetic that i am...,sadness,i am now turning and i feel pathetic that i am...,"['i', 'am', 'now', 'turning', 'and', 'i', 'fee...","['I', 'be', 'now', 'turn', 'and', 'I', 'feel',...","['turn', 'feel', 'pathetic', 'wait', 'table', ...",['pathetic'],"['turn', 'feel', 'pathetic', 'wait', 'table', ...",20
15997,15997,i feel strong and good overall,joy,i feel strong and good overall,"['i', 'feel', 'strong', 'and', 'good', 'overall']","['I', 'feel', 'strong', 'and', 'good', 'overall']","['feel', 'strong', 'good', 'overall']","['strong', 'good']","['feel', 'strong', 'good', 'overall']",6
15998,15998,i feel like this was such a rude comment and i...,anger,i feel like this was such a rude comment and i...,"['i', 'feel', 'like', 'this', 'was', 'such', '...","['I', 'feel', 'like', 'this', 'be', 'such', 'a...","['feel', 'like', 'rude', 'comment', 'm', 'glad...","['rude', 'glad']","['feel', 'like', 'rude', 'comment', 'm', 'glad...",15
15999,15999,i know a lot but i feel so stupid because i ca...,sadness,i know a lot but i feel so stupid because i ca...,"['i', 'know', 'a', 'lot', 'but', 'i', 'feel', ...","['I', 'know', 'a', 'lot', 'but', 'I', 'feel', ...","['know', 'lot', 'feel', 'stupid', 'portray']",['stupid'],"['know', 'lot', 'feel', 'stupid', 'not_portray']",15


In [15]:

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nEmotion distribution:")
print(df["emotion"].value_counts())


Dataset shape: (16000, 10)

Column names:
['Unnamed: 0', 'text', 'emotion', 'doc', 'tokens', 'lemmas', 'clean_tokens', 'adjectives', 'clean_tokens_neg', 'word_count']

Emotion distribution:
emotion
joy         5362
sadness     4666
anger       2159
fear        1937
love        1304
surprise     572
Name: count, dtype: int64
